# CNNs avec PyTorch Lightning sur des données de paysage

Dans ces travaux pratiques, nous classons les photos du jeu des paysages (Intel Image Classification, 6 classes) :

1. avec le CNN des slides, puis un CNN plus profond, que vous écrivez ;
2. en augmentant les données avec `torchvision.transforms.v2` ;
3. avec trois architectures historiques : LeNet, AlexNet et un bloc Inception ;
4. en analysant les erreurs sur l'ensemble de test ;
5. par transfert d'apprentissage : EfficientNetV2-B0 pré-entraîné sur ImageNet (bibliothèque `timm`), gelé puis fine-tuné (partie « Transfert d'apprentissage ») ;
6. avec un *Vision Transformer* pré-entraîné (partie « Transfert d'apprentissage avec des transformeurs »).

Les parties 5 et 6 vont avec le chapitre « Transfer learning » des slides : arrêtez-vous avant elles si vous n'en êtes qu'au chapitre sur les CNN.

## Vérification de l'utilisation de GPU

Allez dans le menu `Exécution > Modifier le type d'exécution` et vérifiez que l'accélérateur matériel est un GPU : sans lui, chaque entraînement de ce TP prendrait des heures.

In [ ]:
!nvidia-smi

## Téléchargement du jeu des paysages

In [ ]:
!wget -q https://github.com/shuuchuu/datasets/raw/refs/heads/main/ghsparse.sh -O /tmp/ghsparse.sh && bash /tmp/ghsparse.sh landscape
!ls landscape
print("***")
!ls -l landscape/seg_train
print("***")
!ls -l landscape/seg_pred

## Installation et imports

In [ ]:
!pip install -q lightning torchmetrics timm

In [ ]:
import itertools
import pathlib

import lightning
import matplotlib.pyplot as plt
import numpy
import pandas
import PIL.Image
import seaborn
import sklearn.metrics
import timm
import torch
import torchmetrics
import tqdm.notebook
import transformers
from lightning.pytorch.callbacks import TQDMProgressBar
from lightning.pytorch.loggers import CSVLogger
from lightning.pytorch.utilities.model_summary import ModelSummary
from torch import nn
from torch.utils.data import DataLoader, Dataset
from torchvision.transforms import v2

# Mêmes tirages aléatoires à chaque exécution (PyTorch, NumPy, Python) :
# des résultats reproductibles
lightning.seed_everything(0)

## Préparation des données

`get_images` lit les images d'un dossier, sous-dossier par sous-dossier : l'étiquette d'une image est l'indice du nom de son sous-dossier dans `label_names` (0 = `buildings`, …, 5 = `street`). Elle les redimensionne à `size` × `size` pixels et les garde toutes en mémoire, dans un tenseur d'entiers de 0 à 255, d'un octet chacun. Les 14 034 images d'entraînement y prennent 0,95 Go en 150 × 150, puis 1,6 Go en 192 × 192 et 2,1 Go en 224 × 224, les tailles du transfert d'apprentissage : la RAM de Colab (12,7 Go) les contient.

`LandscapeDataset` ne convertit une image en flottants dans [0, 1] qu'au moment de la lire, batch par batch : en flottants de 4 octets, le jeu entier prendrait quatre fois plus de place. Il lui applique aussi, pour l'entraînement, une transformation d'augmentation facultative (partie « Augmentation de données »).

`get_dataloaders` garde 30 % des images d'entraînement pour la validation, et `make_loader` sert pour l'ensemble de test.

In [ ]:
INPUT_SHAPE = (3, 150, 150)


label_names = ["buildings", "forest", "glacier", "mountain", "sea", "street"]
label_to_index = {l: i for i, l in enumerate(label_names)}


def get_images(
  dir_path: pathlib.Path,
  size: int = INPUT_SHAPE[1],
  shuffle: bool = True,
  create_labels: bool = True,
) -> tuple[torch.Tensor, torch.Tensor] | torch.Tensor:
  images = []
  if create_labels:
    labels = []

  # On parcourt les sous-dossiers de la racine : chacun correspond à une classe
  for subdir_path in tqdm.notebook.tqdm(
    list(dir_path.iterdir()), desc="Traitement des dossiers"
  ):
    dir_name = subdir_path.name

    if create_labels:
      # L'étiquette est l'indice du nom du dossier
      label = label_to_index.get(dir_name)

    # On ajoute chaque image du dossier courant à notre jeu de données
    for image_path in tqdm.notebook.tqdm(
      list(subdir_path.iterdir()), desc=f"Dossier {dir_name}", leave=False
    ):
      # Pillow charge l'image, en RVB, à la taille demandée
      images.append(
        numpy.array(PIL.Image.open(image_path).convert("RGB").resize((size, size)))
      )
      if create_labels:
        labels.append(label)

  # PyTorch attend les canaux en deuxième position : on passe donc de la forme
  # (N, H, W, C) à la forme (N, C, H, W)
  images = torch.from_numpy(numpy.array(images)).permute(0, 3, 1, 2).contiguous()
  if create_labels:
    labels = torch.tensor(labels)

  if shuffle:
    perm = torch.randperm(images.shape[0])
    images = images[perm]
    if create_labels:
      labels = labels[perm]

  if create_labels:
    return images, labels
  else:
    return images

In [ ]:
class LandscapeDataset(Dataset):
  """Dataset d'images, converties en flottants dans [0, 1] à la volée.

  `augmentation`, une transformation de torchvision (pour l'entraînement
  seulement), s'applique à chaque lecture d'une image : chaque époque en voit
  une variante différente.
  """

  def __init__(
    self,
    images: torch.Tensor,
    labels: torch.Tensor | None = None,
    augmentation: v2.Transform | None = None,
  ) -> None:
    self.images = images
    self.labels = labels
    self.augmentation = augmentation

  def __len__(self) -> int:
    return self.images.shape[0]

  def __getitem__(self, index: int) -> torch.Tensor | tuple[torch.Tensor, torch.Tensor]:
    image = self.images[index]
    if self.augmentation is not None:
      image = self.augmentation(image)
    image = image.float() / 255.0
    if self.labels is None:
      return image
    return image, self.labels[index]


def make_loader(
  images: torch.Tensor,
  labels: torch.Tensor | None = None,
  batch_size: int = 32,
  shuffle: bool = False,
  augmentation: v2.Transform | None = None,
) -> DataLoader:
  # Les images sont déjà en mémoire : le processus principal lit lui-même les
  # lots, sans démarrer de processus pour chacun des entraînements du TP
  return DataLoader(
    LandscapeDataset(images, labels, augmentation),
    batch_size=batch_size,
    shuffle=shuffle,
  )


def get_dataloaders(
  images: torch.Tensor,
  labels: torch.Tensor,
  batch_size: int = 32,
  validation_split: float = 0.3,
  augmentation: v2.Transform | None = None,
) -> tuple[DataLoader, DataLoader]:
  """Sépare les données entre entraînement et validation.

  L'augmentation ne s'applique qu'aux images d'entraînement.
  """
  n_validation = int(images.shape[0] * validation_split)
  n_train = images.shape[0] - n_validation
  return (
    make_loader(
      images[:n_train],
      labels[:n_train],
      batch_size,
      shuffle=True,
      augmentation=augmentation,
    ),
    make_loader(images[n_train:], labels[n_train:], batch_size),
  )


def to_displayable(image: torch.Tensor) -> numpy.ndarray:
  """Passe d'un tenseur (C, H, W) à un tableau (H, W, C) affichable."""
  return image.permute(1, 2, 0).numpy()

## Regardons les données

In [ ]:
images, labels = get_images(pathlib.Path("landscape") / "seg_train")

In [ ]:
print(f"Forme des images : {tuple(images.shape)}")
print(f"Forme des étiquettes : {tuple(labels.shape)}")

counts = torch.bincount(labels, minlength=len(label_names))
plt.bar(label_names, counts.numpy())
plt.title("Nombre d'images par classe (entraînement et validation)")
plt.ylabel("Nombre d'images")
plt.xlabel("Classe")
plt.show()

In [ ]:
# 25 images tirées au hasard, sans remise (pas deux fois la même image)
random_indexes = numpy.random.choice(images.shape[0], size=(5, 5), replace=False)

# figsize agrandit la figure, petite par défaut
_, ax = plt.subplots(5, 5, figsize=(15, 15))
for i, j in itertools.product(range(5), repeat=2):
  img_index = random_indexes[i, j]
  ax[i, j].imshow(to_displayable(images[img_index]))
  ax[i, j].set_title(f"Image {img_index} ({label_names[int(labels[img_index])]})")
  ax[i, j].axis("off")
plt.show()

## Création du modèle

`ImageClassifier` enveloppe chacun de nos modèles dans un `LightningModule` : la perte d'entropie croisée, l'optimiseur AdamW et la justesse (*accuracy*) en entraînement, validation et test. `make_trainer` crée le `Trainer` qui entraîne puis évalue un modèle, en écrivant ses métriques dans un fichier CSV.

Le premier modèle reprend les couches du `ConvNet` des slides (« Définir un modèle avec `torch.nn` »), pour nos images 150 × 150 et nos 6 classes : deux blocs convolution + ReLU + pooling, puis une couche dense de sortie. Les convolutions 3 × 3 avec `padding=1` gardent la taille des cartes, et chaque pooling la divise par 2, arrondie en dessous : 150 → 75 → 37.

In [ ]:
class ImageClassifier(lightning.LightningModule):
  """Enveloppe Lightning commune à tous nos modèles de classification.

  Le modèle encapsulé renvoie des logits : le softmax est appliqué par la
  fonction de perte pendant l'apprentissage, et explicitement au moment de la
  prédiction.
  """

  def __init__(
    self,
    model: nn.Module,
    learning_rate: float = 1e-4,
    weight_decay: float = 0.0,
    num_classes: int = len(label_names),
    input_size: int = INPUT_SHAPE[1],
  ) -> None:
    super().__init__()
    self.save_hyperparameters(ignore=["model"])
    self.model = model
    # Une entrée d'exemple permet à Lightning d'afficher la forme des tenseurs
    # d'entrée et de sortie de chaque couche dans le résumé du modèle
    self.example_input_array = torch.zeros(1, INPUT_SHAPE[0], input_size, input_size)
    # torchmetrics demande une instance de métrique par étape
    self.accuracies = nn.ModuleDict(
      {
        f"{stage}_accuracy": torchmetrics.Accuracy(
          task="multiclass", num_classes=num_classes
        )
        for stage in ("train", "val", "test")
      }
    )

  def forward(self, images: torch.Tensor) -> torch.Tensor:
    return self.model(images)

  def _step(self, batch: tuple[torch.Tensor, torch.Tensor], stage: str) -> torch.Tensor:
    images, labels = batch
    logits = self(images)
    loss = nn.functional.cross_entropy(logits, labels)
    accuracy = self.accuracies[f"{stage}_accuracy"]
    accuracy(logits, labels)
    self.log(f"{stage}_loss", loss, on_step=False, on_epoch=True, prog_bar=True)
    self.log(f"{stage}_accuracy", accuracy, on_step=False, on_epoch=True, prog_bar=True)
    return loss

  def training_step(
    self, batch: tuple[torch.Tensor, torch.Tensor], batch_index: int
  ) -> torch.Tensor:
    return self._step(batch, "train")

  def validation_step(
    self, batch: tuple[torch.Tensor, torch.Tensor], batch_index: int
  ) -> torch.Tensor:
    return self._step(batch, "val")

  def test_step(
    self, batch: tuple[torch.Tensor, torch.Tensor], batch_index: int
  ) -> torch.Tensor:
    return self._step(batch, "test")

  def predict_step(
    self, batch: torch.Tensor | tuple[torch.Tensor, ...], batch_index: int
  ) -> torch.Tensor:
    images = batch[0] if isinstance(batch, (list, tuple)) else batch
    return self(images).softmax(dim=-1).cpu()

  def configure_optimizers(self) -> torch.optim.Optimizer:
    # AdamW applique la weight decay directement aux poids (voir les slides) ;
    # les paramètres gelés (requires_grad=False) n'ont pas de gradient et sont
    # ignorés par l'optimiseur
    return torch.optim.AdamW(
      self.parameters(),
      lr=self.hparams.learning_rate,
      weight_decay=self.hparams.weight_decay,
    )


def make_trainer(max_epochs: int, name: str) -> lightning.Trainer:
  """Crée un Trainer qui journalise ses métriques dans un fichier CSV."""
  return lightning.Trainer(
    max_epochs=max_epochs,
    accelerator="auto",
    devices=1,
    logger=CSVLogger("logs", name=name),
    # La barre de progression de tqdm, plutôt que celle de rich que Lightning
    # choisit quand rich est installé (Colab) : elle a figé des entraînements
    callbacks=[TQDMProgressBar()],
    enable_checkpointing=False,
    num_sanity_val_steps=0,
    log_every_n_steps=10,
  )

In [ ]:
# Les couches du ConvNet des slides, dans un nn.Sequential
model = nn.Sequential(
  nn.Conv2d(3, 16, kernel_size=3, padding=1),  # 16 filtres 3 × 3
  nn.ReLU(),
  nn.MaxPool2d(2),  # fenêtres 2 × 2, pas de 2 : 150 → 75
  nn.Conv2d(16, 32, kernel_size=3, padding=1),
  nn.ReLU(),
  nn.MaxPool2d(2),  # 75 → 37
  nn.Flatten(),  # 32 cartes 37 × 37 → un vecteur de 43 808 valeurs
  nn.Linear(32 * 37 * 37, 6),  # un logit par classe (softmax dans la perte)
)

# L'enveloppe Lightning définit la fonction de perte et l'optimiseur
classifier = ImageClassifier(model, learning_rate=1e-3)

# Résumé du modèle : chaque couche, ses paramètres, ses formes d'entrée et de sortie
print(ModelSummary(classifier, max_depth=-1))

## Pouvez-vous expliquer les différents nombres de paramètres ?

Retrouvez par le calcul le nombre de paramètres de chaque couche du résumé. Pourquoi certaines couches n'en ont-elles aucun ?

### Solution

- `model.0` : chacun des 16 filtres a 3 × 3 poids par canal d'entrée, sur 3 canaux (RVB), plus un biais : (3 × 3 × 3 + 1) × 16 = 448.
- `model.3` : 32 filtres, qui couvrent chacun les 16 canaux de la couche précédente : (3 × 3 × 16 + 1) × 32 = 4 640.
- `model.7` : 32 × 37 × 37 = 43 808 entrées et 6 sorties, plus 6 biais : 43 808 × 6 + 6 = 262 854.
- Les ReLU, les poolings et `Flatten` n'apprennent rien : ils appliquent une opération fixe (annuler les valeurs négatives, prendre un maximum, aplatir). Les métriques de justesse (`accuracies`) n'ont pas de paramètres non plus.

Total : 267 942 paramètres, dont 98 % dans la couche dense : `Flatten` lui passe chaque position de chaque carte.

## Apprentissage

Entraînons ce modèle une seule époque, pour vérifier que tout fonctionne : `get_dataloaders` sépare les données, `make_trainer` crée le `Trainer`, et `trainer.fit` entraîne le modèle en l'évaluant sur la validation à la fin de chaque époque.

La cellule définit ensuite deux fonctions pour la suite :

- `plot_metrics(trainer)` trace les courbes de justesse et de perte d'un entraînement ;
- `train(model, max_epochs, name)` regroupe ces étapes (enveloppe, données par batchs de 128, entraînement, courbes) et renvoie l'enveloppe entraînée et son `Trainer`, qui sert ensuite à l'évaluer.

In [ ]:
train_loader, validation_loader = get_dataloaders(images, labels)
trainer = make_trainer(max_epochs=1, name="cnn_des_slides")
trainer.fit(classifier, train_loader, validation_loader)


def plot_metrics(trainer: lightning.Trainer) -> None:
  """Trace la justesse et la perte, en entraînement et en validation."""
  # Les métriques d'entraînement et de validation ne sont pas écrites sur les
  # mêmes lignes du fichier CSV : on les regroupe par époque
  metrics = (
    pandas.read_csv(pathlib.Path(trainer.logger.log_dir) / "metrics.csv")
    .groupby("epoch")
    .mean(numeric_only=True)
    .dropna(subset=["train_loss", "val_loss"])
  )
  for metric, name in (("accuracy", "Justesse"), ("loss", "Perte")):
    plt.plot(metrics.index, metrics[f"train_{metric}"], label="Entraînement")
    plt.plot(metrics.index, metrics[f"val_{metric}"], label="Validation")
    plt.title(f"{name} du modèle")
    plt.ylabel(name)
    plt.xlabel("Époque")
    plt.legend()
    plt.show()


def train(
  model: nn.Module,
  max_epochs: int,
  name: str,
  learning_rate: float = 1e-4,
  augmentation: v2.Transform | None = None,
) -> tuple[ImageClassifier, lightning.Trainer]:
  """Entraîne un modèle sur les images 150 × 150, puis trace ses courbes.

  Renvoie l'enveloppe entraînée et le Trainer, qui sert ensuite à l'évaluer.
  """
  classifier = ImageClassifier(model, learning_rate=learning_rate)
  print(ModelSummary(classifier, max_depth=-1))
  train_loader, validation_loader = get_dataloaders(
    images, labels, augmentation=augmentation
  )
  trainer = make_trainer(max_epochs, name)
  trainer.fit(classifier, train_loader, validation_loader)
  plot_metrics(trainer)
  return classifier, trainer

## Améliorez cette performance

Le CNN des slides ne voit que de petits motifs, à travers deux convolutions : trop peu pour reconnaître une scène. Écrivez une fonction `deep_cnn()` qui renvoie un CNN plus profond, avec des poids neufs à chaque appel, puis entraînez-le avec `train` :

- des blocs convolution + ReLU + pooling répétés, jusqu'à des cartes de quelques pixels de côté ;
- du *dropout* avant les couches denses de la tête, contre le surapprentissage ;
- entre 10 et 20 époques, chacune en moins d'une minute sur le GPU de Colab.

Objectif : plus de 70 % de justesse en validation. La solution atteint environ 85 % en 15 époques, de 26 secondes chacune sur une RTX A4000. `deep_cnn()` resservira pour l'augmentation de données.

**Documentation utile** :

- https://docs.pytorch.org/docs/stable/generated/torch.nn.Dropout.html
- https://docs.pytorch.org/docs/stable/generated/torch.nn.MaxPool2d.html
- https://docs.pytorch.org/docs/stable/nn.init.html#torch.nn.init.orthogonal_
- https://docs.pytorch.org/docs/stable/nn.init.html#torch.nn.init.xavier_uniform_

In [ ]:
def deep_cnn() -> nn.Sequential: ...


# Votre code ici

### Conseils supplémentaires

- Un bloc : `nn.Conv2d(in_channels, 200, kernel_size=3, padding="same")`, `nn.ReLU()`, puis `nn.MaxPool2d(2, ceil_mode=True)`. La première convolution prend les 3 canaux de l'image, les suivantes les 200 de la précédente.
- Avec `padding="same"`, la convolution garde la taille des cartes et seul le pooling la divise par 2, arrondie au-dessus avec `ceil_mode=True` : 150 → 75 → 38 → 19 → 10 → 5 → 3 en 6 blocs. Une 7ᵉ convolution, sans pooling, termine sur 200 cartes 3 × 3.
- La solution initialise les poids de chaque convolution avec `nn.init.orthogonal_(convolution.weight)`, et ceux des couches denses avec `nn.init.xavier_uniform_` (l'initialisation de Glorot, celle de Keras), leurs biais à zéro : avec l'initialisation par défaut de PyTorch, plus petite, le signal diminue à chaque couche de la tête et le modèle apprend plus lentement.
- Tête : `nn.Flatten()`, qui donne 200 × 3 × 3 = 1 800 valeurs, puis trois couches `nn.Linear` de 200, 100 et 50 neurones suivies chacune d'une ReLU, et la sortie `nn.Linear(50, 6)`. Chacune de ces quatre couches est précédée de `nn.Dropout(0.2)`.
- Entraînement : `train(deep_cnn(), max_epochs=15, name="cnn_profond")`, avec le taux d'apprentissage par défaut de `train` (1e-4).

Un squelette à copier dans la cellule de l'exercice, en remplaçant chaque `...` :

```python
def conv_block(in_channels: int) -> list[nn.Module]:
  convolution = nn.Conv2d(in_channels, 200, kernel_size=3, padding="same")
  nn.init.orthogonal_(convolution.weight)
  return [convolution, nn.ReLU()]


def deep_cnn() -> nn.Sequential:
  blocks = []
  in_channels = 3
  for _ in range(6):
    blocks += [*conv_block(in_channels), nn.MaxPool2d(...)]
    in_channels = 200
  model = nn.Sequential(
    *blocks,
    *conv_block(200),
    nn.Flatten(),
    nn.Dropout(0.2),
    nn.Linear(..., 200),
    nn.ReLU(),
    ...,
    nn.Linear(50, 6),
  )
  for layer in model:
    if isinstance(layer, nn.Linear):
      nn.init.xavier_uniform_(layer.weight)
      nn.init.zeros_(layer.bias)
  return model


classifier, trainer = train(deep_cnn(), max_epochs=..., name="cnn_profond")
```

### Solution

In [ ]:
def conv_block(in_channels: int) -> list[nn.Module]:
  """Convolution 3 × 3 à 200 filtres, qui garde la taille des cartes, puis ReLU."""
  convolution = nn.Conv2d(in_channels, 200, kernel_size=3, padding="same")
  nn.init.orthogonal_(convolution.weight)
  return [convolution, nn.ReLU()]


def deep_cnn() -> nn.Sequential:
  """Le CNN profond de la solution, avec des poids neufs à chaque appel."""
  blocks = []
  in_channels = 3
  for _ in range(6):  # cartes de 150 → 75 → 38 → 19 → 10 → 5 → 3 pixels de côté
    blocks += [*conv_block(in_channels), nn.MaxPool2d(2, ceil_mode=True)]
    in_channels = 200
  model = nn.Sequential(
    *blocks,
    *conv_block(200),
    nn.Flatten(),  # 200 cartes 3 × 3 → 1 800 valeurs
    nn.Dropout(0.2),
    nn.Linear(200 * 3 * 3, 200),
    nn.ReLU(),
    nn.Dropout(0.2),
    nn.Linear(200, 100),
    nn.ReLU(),
    nn.Dropout(0.2),
    nn.Linear(100, 50),
    nn.ReLU(),
    nn.Dropout(0.2),
    nn.Linear(50, 6),
  )
  # Initialisation de Glorot pour les couches denses, celle de Keras : avec
  # celle de PyTorch, plus petite, le signal diminue à chacune des quatre
  # couches de la tête, et le modèle apprend plus lentement
  for layer in model:
    if isinstance(layer, nn.Linear):
      nn.init.xavier_uniform_(layer.weight)
      nn.init.zeros_(layer.bias)
  return model

In [ ]:
classifier, trainer = train(deep_cnn(), max_epochs=15, name="cnn_profond")

## Augmentation de données

Quand la justesse d'entraînement continue de monter alors que celle de validation stagne, le modèle surapprend. L'augmentation de données (slide « Transformations et augmentation de données ») lui présente à chaque époque des variantes des images d'entraînement. Écrivez avec `torchvision.transforms.v2` une transformation `augmentation` qui :

- recadre une image au hasard sur 70 à 100 % de sa surface, puis la ramène à 150 × 150 pixels ;
- la retourne horizontalement une fois sur deux.

Passée à `train(..., augmentation=augmentation)`, elle ne s'applique qu'aux images d'entraînement : celles de validation restent intactes. Entraînez la même architecture, avec des poids neufs (`deep_cnn()`), pendant 15 époques. L'écart entre les courbes d'entraînement et de validation se réduit-il ?

**Documentation utile** :

- https://docs.pytorch.org/vision/stable/generated/torchvision.transforms.v2.RandomResizedCrop.html

In [ ]:
# Votre code ici

### Conseils supplémentaires

- `v2.Compose([...])` enchaîne des transformations, dans l'ordre de la liste.
- `v2.RandomResizedCrop(150, scale=(0.7, 1.0), antialias=True)` : la taille de sortie, puis la part de la surface gardée.
- `v2.RandomHorizontalFlip()` retourne l'image avec une probabilité de 0,5 par défaut.
- Les transformations travaillent sur les entiers de 0 à 255 des images, que `LandscapeDataset` convertit ensuite en flottants.

Un squelette à copier dans la cellule de l'exercice, en remplaçant chaque `...` :

```python
augmentation = v2.Compose([v2.RandomResizedCrop(...), v2.RandomHorizontalFlip()])
classifier, trainer = train(
  deep_cnn(), max_epochs=15, name="cnn_profond_augmente", augmentation=...
)
```

### Solution

In [ ]:
augmentation = v2.Compose(
  [
    # recadrage sur 70 à 100 % de la surface, ramené à 150 × 150
    v2.RandomResizedCrop(150, scale=(0.7, 1.0), antialias=True),
    v2.RandomHorizontalFlip(),  # miroir, une image sur deux
  ]
)
classifier, trainer = train(
  deep_cnn(), max_epochs=15, name="cnn_profond_augmente", augmentation=augmentation
)

Chaque époque voit des variantes différentes des images : la justesse d'entraînement monte moins vite (85 % environ après 15 époques, contre 91 % sans augmentation) et reste sous celle de validation presque jusqu'au bout, alors que sans augmentation elle la dépassait dès la 7ᵉ époque environ. La validation finit vers 85 % dans les deux cas, mais l'écart entre les deux courbes a disparu. Le modèle doit apprendre des caractéristiques qui résistent au cadrage et à l'orientation, au lieu de mémoriser les images d'entraînement. L'évaluation sur le test et l'analyse d'erreur portent sur ce modèle : `classifier` et son `trainer`.

## Architectures historiques

Trois architectures du chapitre sur les CNN, à entraîner puis comparer au CNN profond : leur code est donné. Chacune a ses propres noms de variables, et `classifier` et `trainer` restent ceux du CNN profond augmenté.

### LeNet

[LeNet-5](https://en.wikipedia.org/wiki/LeNet) (1998), conçu pour des chiffres manuscrits de 32 × 32 pixels : des convolutions 5 × 5, des activations sigmoïdes et des poolings par moyenne.

In [ ]:
le_net = nn.Sequential(
  nn.Conv2d(3, 6, kernel_size=5, padding="same"),
  nn.Sigmoid(),
  nn.AvgPool2d(2),  # 150 → 75
  nn.Conv2d(6, 16, kernel_size=5),  # 75 → 71
  nn.Sigmoid(),
  nn.AvgPool2d(2),  # 71 → 35
  nn.Flatten(),
  nn.Linear(16 * 35 * 35, 120),
  nn.Sigmoid(),
  nn.Linear(120, 84),
  nn.Sigmoid(),
  nn.Linear(84, 6),
)
le_net_classifier, le_net_trainer = train(le_net, max_epochs=15, name="le_net")

### AlexNet

[AlexNet](https://en.wikipedia.org/wiki/AlexNet) (2012), vainqueur d'ImageNet : une première convolution 11 × 11 de pas 4, des ReLU, du *dropout* dans une tête de deux couches de 4 096 neurones.

In [ ]:
alex_net = nn.Sequential(
  nn.Conv2d(3, 96, kernel_size=11, stride=4),  # 150 → 35
  nn.ReLU(),
  nn.MaxPool2d(3, stride=2),  # 35 → 17
  nn.Conv2d(96, 256, kernel_size=5, padding="same"),
  nn.ReLU(),
  nn.MaxPool2d(3, stride=2),  # 17 → 8
  nn.Conv2d(256, 384, kernel_size=3, padding="same"),
  nn.ReLU(),
  nn.Conv2d(384, 384, kernel_size=3, padding="same"),
  nn.ReLU(),
  nn.Conv2d(384, 256, kernel_size=3, padding="same"),
  nn.ReLU(),
  nn.MaxPool2d(3, stride=2),  # 8 → 3
  nn.Flatten(),
  nn.Linear(256 * 3 * 3, 4096),
  nn.ReLU(),
  nn.Dropout(0.5),
  nn.Linear(4096, 4096),
  nn.ReLU(),
  nn.Dropout(0.5),
  nn.Linear(4096, 6),
)
alex_net_classifier, alex_net_trainer = train(alex_net, max_epochs=15, name="alex_net")

### Bloc Inception

Un bloc [Inception](https://en.wikipedia.org/wiki/Inception_(deep_learning_architecture)) (GoogLeNet, 2014) : quatre branches en parallèle (convolutions 1 × 1, 3 × 3, 5 × 5 et pooling), concaténées sur l'axe des canaux. Un réseau complet en empile plusieurs. Le flux de données n'est plus séquentiel : le bloc est un [module PyTorch](https://docs.pytorch.org/docs/stable/notes/modules.html) personnalisé, dont la méthode `forward` combine les branches.

In [ ]:
class InceptionBlock(nn.Module):
  """Bloc Inception : quatre branches concaténées sur l'axe des canaux."""

  def __init__(self, in_channels: int, out_channels: int = 64) -> None:
    super().__init__()

    # conv 1 × 1
    self.conv11 = nn.Sequential(nn.Conv2d(in_channels, out_channels, 1), nn.ReLU())

    # conv 1 × 1 puis conv 3 × 3
    self.conv33 = nn.Sequential(
      nn.Conv2d(in_channels, out_channels, 1),
      nn.ReLU(),
      nn.Conv2d(out_channels, out_channels, 3, padding="same"),
      nn.ReLU(),
    )

    # conv 1 × 1 puis conv 5 × 5
    self.conv55 = nn.Sequential(
      nn.Conv2d(in_channels, out_channels, 1),
      nn.ReLU(),
      nn.Conv2d(out_channels, out_channels, 5, padding="same"),
      nn.ReLU(),
    )

    # max pooling 3 × 3 de pas 1 puis conv 1 × 1
    self.max_pool = nn.Sequential(
      nn.MaxPool2d(3, 1, padding=1), nn.Conv2d(in_channels, out_channels, 1), nn.ReLU()
    )

  def forward(self, inputs: torch.Tensor) -> torch.Tensor:
    # concaténation sur l'axe des canaux : 4 × 64 = 256 canaux
    return torch.cat(
      [
        self.conv11(inputs),
        self.conv33(inputs),
        self.conv55(inputs),
        self.max_pool(inputs),
      ],
      dim=1,
    )


inception = nn.Sequential(
  InceptionBlock(INPUT_SHAPE[0]),
  nn.Flatten(),
  nn.Linear(4 * 64 * 150 * 150, 6),
)
inception_classifier, inception_trainer = train(
  inception, max_epochs=10, name="inception"
)

## Évaluation des performances sur l'ensemble de test

Le dossier `seg_test` contient 3 000 images que le modèle n'a vues ni à l'entraînement, ni en validation : sa justesse sur elles estime honnêtement celle qu'il aura sur de nouvelles photos. La méthode `test` du `Trainer` qui a entraîné le modèle la calcule. `test_accuracies` garde la justesse en test de chaque approche, pour les comparer à la fin du TP.

In [ ]:
test_images, test_labels = get_images(pathlib.Path("landscape") / "seg_test")
test_loader = make_loader(test_images, test_labels, batch_size=128)

# Le Trainer qui a entraîné le modèle est aussi celui qui l'évalue
results = trainer.test(classifier, dataloaders=test_loader)
test_accuracies = {"CNN profond, données augmentées": results[0]["test_accuracy"]}


def predict_probabilities(
  trainer: lightning.Trainer,
  classifier: ImageClassifier,
  images: torch.Tensor,
  batch_size: int = 128,
) -> torch.Tensor:
  """Renvoie la probabilité attribuée à chaque classe pour chaque image."""
  return torch.cat(
    trainer.predict(classifier, make_loader(images, batch_size=batch_size))
  )

## Analyse d'erreur

On affiche la matrice de confusion, puis des images mal classées.

In [ ]:
def analyze_preds(preds: torch.Tensor, labels: torch.Tensor) -> None:
  confusion_matrix = sklearn.metrics.confusion_matrix(labels, preds)
  seaborn.heatmap(
    confusion_matrix,
    annot=True,
    fmt="d",
    cmap="rocket_r",
    xticklabels=label_names,
    yticklabels=label_names,
  )
  plt.title("Matrice de confusion")
  plt.xlabel("Classe prédite")
  plt.ylabel("Classe réelle")
  plt.show()

  seaborn.countplot(x=[label_names[pred] for pred in preds.tolist()])
  plt.title("Décomptes des classes prédites")
  plt.ylabel("Décompte")
  plt.xlabel("Classe")
  plt.show()


test_pred = predict_probabilities(trainer, classifier, test_images).argmax(dim=-1)
analyze_preds(test_pred, test_labels)

In [ ]:
def plot_mistakes(predicted_class: str, true_class: str) -> None:
  """Affiche jusqu'à 25 images de test de true_class prédites predicted_class."""
  print(f"Prédiction : {predicted_class}, classe réelle : {true_class}")
  mistakes = test_images[
    (test_pred == label_names.index(predicted_class))
    & (test_labels == label_names.index(true_class))
  ]
  random_indexes = numpy.random.choice(
    mistakes.shape[0], size=min(mistakes.shape[0], 25), replace=False
  )

  _, ax = plt.subplots(5, 5, figsize=(15, 15))
  grid_indexes = list(itertools.product(range(5), repeat=2))
  for i, j in grid_indexes:
    ax[i, j].axis("off")
  for img_index, (i, j) in zip(random_indexes, grid_indexes):
    ax[i, j].imshow(to_displayable(mistakes[img_index]))
  plt.show()

In [ ]:
# Images prédites glacier alors qu'elles sont étiquetées montagne
plot_mistakes("glacier", "mountain")

In [ ]:
# Images prédites glacier alors qu'elles sont étiquetées mer
plot_mistakes("glacier", "sea")

In [ ]:
# Images prédites bâtiments alors qu'elles sont étiquetées rue
plot_mistakes("buildings", "street")

## Transfert d'apprentissage

On réutilise un réseau pré-entraîné sur ImageNet, EfficientNetV2-B0 (bibliothèque `timm`), comme extracteur de caractéristiques : ses poids sont gelés, et on n'entraîne qu'une nouvelle tête de classification pour nos 6 classes (slides du chapitre « Transfer learning »).

Les poids pré-entraînés attendent les images telles qu'à leur entraînement : `timm.data.resolve_model_data_config` donne leur taille, ainsi que la moyenne et l'écart type de leur normalisation. Ce réseau a été entraîné sur des images 192 × 192 : on recharge nos images à cette taille.

Avec `num_classes=0`, `timm` crée le réseau sans sa tête ImageNet (1 000 classes) : il renvoie, pour chaque image, la moyenne de ses 1 280 cartes de caractéristiques finales (pooling global).

In [ ]:
base_model = timm.create_model(
  "tf_efficientnetv2_b0.in1k", pretrained=True, num_classes=0
)
# Taille d'entrée, interpolation, moyenne et écart type de normalisation
data_config = timm.data.resolve_model_data_config(base_model)
print(data_config)

transfer_size = data_config["input_size"][-1]
images, labels = get_images(pathlib.Path("landscape") / "seg_train", size=transfer_size)
test_images, test_labels = get_images(
  pathlib.Path("landscape") / "seg_test", size=transfer_size
)
test_loader = make_loader(test_images, test_labels, batch_size=128)

Dans les slides, la normalisation est une transformation (`v2.Normalize`) appliquée par le `Dataset`. Ici, elle est la première couche du modèle (module `Normalize`) : les CNN entraînés depuis zéro et le modèle pré-entraîné partagent ainsi les mêmes `DataLoader`, qui produisent des images dans [0, 1].

Le réseau pré-entraîné est encapsulé dans `Frozen` (slide « Geler le backbone »), qui gèle ses poids **et** le maintient en mode évaluation. `requires_grad=False` ne suffit pas : `trainer.fit` passe le modèle en mode `train()`, ce qui ferait évoluer les statistiques de ses couches de BatchNorm et réactiverait son dropout.

La nouvelle tête est une seule couche dense, des 1 280 caractéristiques vers nos 6 classes (slide « Remplacer la tête »). Elle apprend avec un taux de 1e-3, comme dans les slides.

In [ ]:
class Normalize(nn.Module):
  """Normalise, canal par canal, des images déjà ramenées dans [0, 1]."""

  def __init__(
    self, mean: tuple[float, ...] | float, std: tuple[float, ...] | float
  ) -> None:
    super().__init__()
    self.register_buffer("mean", torch.tensor(mean).reshape(1, -1, 1, 1))
    self.register_buffer("std", torch.tensor(std).reshape(1, -1, 1, 1))

  def forward(self, images: torch.Tensor) -> torch.Tensor:
    return (images - self.mean) / self.std


class Frozen(nn.Module):
  """Encapsule un module gelé, maintenu en mode évaluation."""

  def __init__(self, module: nn.Module) -> None:
    super().__init__()
    self.module = module.eval().requires_grad_(False)

  def train(self, mode: bool = True) -> "Frozen":
    return super().train(False)  # BatchNorm et dropout restent figés

  def forward(self, images: torch.Tensor) -> torch.Tensor:
    return self.module(images)


model = nn.Sequential(
  Normalize(data_config["mean"], data_config["std"]),
  Frozen(base_model),  # un vecteur de 1 280 caractéristiques par image
  nn.Linear(base_model.num_features, 6),  # la nouvelle tête
)

# La weight decay d'AdamW ne touche que les poids qui reçoivent un gradient :
# ceux de la tête, le réseau pré-entraîné étant gelé
classifier = ImageClassifier(
  model, learning_rate=1e-3, weight_decay=0.01, input_size=transfer_size
)

# max_depth=2 : sans le détail des couches d'EfficientNetV2
print(ModelSummary(classifier, max_depth=2))

In [ ]:
train_loader, validation_loader = get_dataloaders(images, labels, batch_size=64)
trainer = make_trainer(max_epochs=10, name="transfert")
trainer.fit(classifier, train_loader, validation_loader)
plot_metrics(trainer)

In [ ]:
results = trainer.test(classifier, dataloaders=test_loader)
test_accuracies["EfficientNetV2-B0 gelé"] = results[0]["test_accuracy"]

test_pred = predict_probabilities(trainer, classifier, test_images).argmax(dim=-1)
analyze_preds(test_pred, test_labels)
plot_mistakes("glacier", "mountain")
plot_mistakes("glacier", "sea")
plot_mistakes("buildings", "street")

### Questions

1. Combien de paramètres ce modèle entraîne-t-il, et combien en gèle-t-il ? (voir son résumé)
2. Comparez sa justesse de test à celle du CNN profond entraîné depuis zéro, et la durée de leurs époques.
3. Les erreurs restantes (matrice de confusion, images mal classées) ressemblent-elles à celles du CNN ?

#### Solution

1. Seule la tête est entraînée : 1 280 × 6 + 6 = 7 686 paramètres, contre 5 858 704 gelés dans EfficientNetV2-B0.
2. Le modèle pré-entraîné dépasse le CNN profond dès les premières époques. Une époque ne coûte presque que le passage avant dans EfficientNetV2 : aucun gradient ne le traverse, puisqu'il est gelé. Les caractéristiques qu'il a apprises sur plus d'un million d'images d'ImageNet se transfèrent bien à nos 14 000 paysages.
3. Les confusions qui restent opposent surtout des classes réellement proches (glacier et montagne, bâtiments et rue) ; beaucoup de ces images sont ambiguës, voire mal annotées.

### Fine-tuning : dégeler le backbone

Une fois la tête entraînée, on peut dégeler EfficientNetV2 et entraîner tout le réseau quelques époques de plus, avec un taux d'apprentissage plus faible pour le backbone que pour la tête (slide « Fine-tuning : dégeler le backbone ») : ses poids sont déjà bons, on veut seulement les ajuster à nos paysages.

`FineTuner` ci-dessous donne au backbone (`model[1]`) son propre taux d'apprentissage. Retirez `Frozen` du modèle (le réseau gelé est `model[1].module`) et dégelez ses poids. Entraînez ensuite 3 époques, avec un taux de 1e-5 pour le backbone et de 1e-4 pour la tête, puis ajoutez la justesse en test à `test_accuracies`. La justesse de validation progresse-t-elle ?

In [ ]:
class FineTuner(ImageClassifier):
  """ImageClassifier dont le backbone (`model[1]`) a son propre taux d'apprentissage."""

  def __init__(self, model: nn.Module, backbone_learning_rate: float, **kwargs) -> None:
    super().__init__(model, **kwargs)
    self.backbone_learning_rate = backbone_learning_rate

  def configure_optimizers(self) -> torch.optim.Optimizer:
    backbone = list(self.model[1].parameters())
    backbone_ids = {id(parameter) for parameter in backbone}
    head = [p for p in self.parameters() if id(p) not in backbone_ids]
    return torch.optim.AdamW(
      [
        {"params": backbone, "lr": self.backbone_learning_rate},
        {"params": head, "lr": self.hparams.learning_rate},
      ],
      weight_decay=self.hparams.weight_decay,
    )

In [ ]:
# Votre code ici

#### Conseils supplémentaires

- `model[1].module.requires_grad_(True)` dégèle EfficientNetV2 et le renvoie : remettez-le à la place de son enveloppe, `model[1] = ...`. Sans `Frozen`, il suit de nouveau `train()` et `eval()`, et ses BatchNorm s'adaptent à nos images.
- `FineTuner(model, backbone_learning_rate=1e-5, learning_rate=1e-4, weight_decay=0.01, input_size=transfer_size)` : `learning_rate` est celui de la tête.
- Un nouveau `Trainer`, `make_trainer(max_epochs=3, name="fine_tuning")`, entraîné sur les mêmes `train_loader` et `validation_loader`, puis `plot_metrics` et `trainer.test`.

Un squelette à copier dans la cellule de l'exercice, en remplaçant chaque `...` :

```python
model[1] = ...
classifier = FineTuner(
  model,
  backbone_learning_rate=...,
  learning_rate=...,
  weight_decay=0.01,
  input_size=transfer_size,
)
trainer = make_trainer(max_epochs=..., name="fine_tuning")
trainer.fit(...)
plot_metrics(trainer)
results = trainer.test(classifier, dataloaders=test_loader)
test_accuracies["EfficientNetV2-B0 fine-tuné"] = results[0]["test_accuracy"]
```

#### Solution

In [ ]:
# Le backbone remplace son enveloppe Frozen, et ses poids sont dégelés
model[1] = model[1].module.requires_grad_(True)
classifier = FineTuner(
  model,
  backbone_learning_rate=1e-5,
  learning_rate=1e-4,
  weight_decay=0.01,
  input_size=transfer_size,
)
trainer = make_trainer(max_epochs=3, name="fine_tuning")
trainer.fit(classifier, train_loader, validation_loader)
plot_metrics(trainer)
results = trainer.test(classifier, dataloaders=test_loader)
test_accuracies["EfficientNetV2-B0 fine-tuné"] = results[0]["test_accuracy"]

La justesse gagne encore un peu, quelques dixièmes de point ici : le backbone s'adapte à nos paysages. Un taux d'apprentissage trop élevé pour le backbone détruirait au contraire ce qu'il a appris sur ImageNet. La suite du TP utilise ce modèle.

## Prédire dans des conditions « réelles »

Le dossier `seg_pred` contient des images sans étiquette : on ne peut pas y mesurer la justesse. On peut en revanche afficher des photos et les probabilités que le modèle attribue à chaque classe.

In [ ]:
pred_images = get_images(
  pathlib.Path("landscape") / "seg_pred", create_labels=False, size=transfer_size
)
print(f"{pred_images.shape[0]} images sans étiquette")

In [ ]:
# 25 images au hasard, sans remise
random_indexes = numpy.random.choice(pred_images.shape[0], size=(5, 5), replace=False)

# Probabilités des classes pour ces 25 images
probabilities = predict_probabilities(
  trainer, classifier, pred_images[random_indexes.ravel()]
).reshape(5, 5, -1)

# Chaque image, puis la distribution de ses prédictions sur la ligne d'en dessous
_, ax = plt.subplots(10, 5, figsize=(30, 45))
for i, j in itertools.product(range(5), repeat=2):
  img_index = random_indexes[i, j]
  predicted_class = label_names[int(probabilities[i, j].argmax())]
  ax[i * 2, j].imshow(to_displayable(pred_images[img_index]))
  ax[i * 2, j].set_title(f"Image {img_index} ({predicted_class})")
  ax[i * 2, j].axis("off")
  ax[i * 2 + 1, j].bar(label_names, probabilities[i, j].numpy())
plt.show()

## Justesse en fonction de la probabilité maximale

La justesse varie-t-elle selon la probabilité maximale que rend le modèle ? Si oui, on peut ne garder que les prédictions sûres, et confier les autres à un humain.

In [ ]:
test_probabilities = predict_probabilities(trainer, classifier, test_images)

In [ ]:
def threshold_accuracy(
  probabilities: torch.Tensor, labels: torch.Tensor, threshold: float
) -> tuple[float, float, float, float]:
  predictions = probabilities.argmax(dim=-1)
  mask_above = probabilities.max(dim=-1).values > threshold
  mask_below = ~mask_above

  n_above = int(mask_above.sum())
  n_below = int(mask_below.sum())

  if n_above:
    above = float((predictions[mask_above] == labels[mask_above]).sum()) / n_above
  else:
    above = 1.0
  if n_below:
    below = float((predictions[mask_below] == labels[mask_below]).sum()) / n_below
  else:
    below = 1.0
  return above, below, n_above / labels.shape[0], n_below / labels.shape[0]


accuracy_above, accuracy_below, ratio_above, ratio_below = threshold_accuracy(
  test_probabilities, test_labels, 0.99
)
print(
  "Justesse pour les prédictions du modèle qui ont une probabilité au "
  f"dessus du seuil ({ratio_above * 100:.2f}% des données) : "
  f"{accuracy_above:.2f}"
)
print(
  "Justesse pour les prédictions du modèle qui ont une probabilité en "
  f"dessous du seuil ({ratio_below * 100:.2f}% des données) : "
  f"{accuracy_below:.2f}"
)

xs = numpy.arange(101) / 100
accuracies = []
ratios_above = []
for x in xs:
  accuracy, _, ratio_above, _ = threshold_accuracy(test_probabilities, test_labels, x)
  accuracies.append(accuracy)
  ratios_above.append(ratio_above)
plt.plot(xs, accuracies, label="Justesse")
plt.plot(xs, ratios_above, label="Proportion de prédictions retenues")
plt.xlabel("Seuil de prédiction")
plt.title("Justesse et proportion de prédictions retenues selon le seuil")
plt.legend()
plt.show()

## Transfert d'apprentissage avec des transformeurs

Même démarche avec un *Vision Transformer* (ViT) pré-entraîné sur ImageNet-21k (14 millions d'images), chargé avec la bibliothèque `transformers`. Le ViT découpe une image 224 × 224 en 196 carrés de 16 × 16 pixels, et ajoute devant eux un token de classification, `[CLS]`. Le dernier état de ce token résume l'image : la tête de classification se branche sur lui.

La cellule suivante charge les prétraitements du ViT (`AutoImageProcessor` : taille, moyenne et écart type) et recharge les images à sa taille. Construisez ensuite `vit_model`, un `nn.Sequential` :

- `Normalize`, avec la moyenne et l'écart type du ViT ;
- le ViT (`transformers.AutoModel`), gelé par `Frozen`, derrière un module `ViTFeatures` à écrire, qui renvoie l'état final du token `[CLS]` ;
- une couche dense de sortie, de la taille de cet état (768) vers nos 6 classes.

Enveloppez-le dans un `ImageClassifier` nommé `classifier`, avec un taux d'apprentissage de 1e-3, comme la tête d'EfficientNetV2.

**Documentation utile** :

- https://huggingface.co/docs/transformers/model_doc/vit#transformers.ViTModel

In [ ]:
VIT = "google/vit-base-patch16-224-in21k"
# Prétraitements attendus par le ViT : taille, moyenne et écart type
processor = transformers.AutoImageProcessor.from_pretrained(VIT)
vit_size = processor.size["height"]
print(vit_size, processor.image_mean, processor.image_std)

images, labels = get_images(pathlib.Path("landscape") / "seg_train", size=vit_size)
test_images, test_labels = get_images(
  pathlib.Path("landscape") / "seg_test", size=vit_size
)
test_loader = make_loader(test_images, test_labels, batch_size=128)

In [ ]:
# Votre code ici

### Conseils supplémentaires

- Chargement : `transformers.AutoModel.from_pretrained(VIT)` (350 Mo à télécharger).
- `ViTFeatures` est un `nn.Module` qui garde le ViT en attribut. Sa méthode `forward` appelle `self.vit(pixel_values=images)`, dont l'attribut `last_hidden_state` donne un vecteur de 768 valeurs par token, de forme `(N, 197, 768)` : le token `[CLS]` est le premier, `[:, 0]`.
- `Normalize(processor.image_mean, processor.image_std)` ramène les pixels de [0, 1] à [-1, 1].
- La taille de l'état du token vaut `vit.config.hidden_size`.
- `ImageClassifier(vit_model, learning_rate=1e-3, input_size=vit_size)`.

Un squelette à copier dans la cellule de l'exercice, en remplaçant chaque `...` :

```python
class ViTFeatures(nn.Module):
  def __init__(self, vit: nn.Module) -> None:
    super().__init__()
    self.vit = vit

  def forward(self, images: torch.Tensor) -> torch.Tensor:
    return ...


vit = transformers.AutoModel.from_pretrained(VIT)
vit_model = nn.Sequential(
  Normalize(...),
  Frozen(...),
  nn.Linear(...),
)
classifier = ImageClassifier(...)
print(ModelSummary(classifier, max_depth=2))
```

### Solution

In [ ]:
class ViTFeatures(nn.Module):
  """Renvoie l'état final du token de classification [CLS] calculé par le ViT."""

  def __init__(self, vit: nn.Module) -> None:
    super().__init__()
    self.vit = vit

  def forward(self, images: torch.Tensor) -> torch.Tensor:
    # (N, 197, 768) : un vecteur par token, [CLS] en premier
    return self.vit(pixel_values=images).last_hidden_state[:, 0]


vit = transformers.AutoModel.from_pretrained(VIT)

vit_model = nn.Sequential(
  Normalize(processor.image_mean, processor.image_std),  # [0, 1] → [-1, 1]
  Frozen(ViTFeatures(vit)),
  nn.Linear(vit.config.hidden_size, 6),
)
classifier = ImageClassifier(vit_model, learning_rate=1e-3, input_size=vit_size)
print(ModelSummary(classifier, max_depth=2))

### Comparaison des trois approches

Entraînez le ViT 5 époques, évaluez-le sur l'ensemble de test, puis comparez les trois approches : CNN entraîné depuis zéro, EfficientNetV2-B0 pré-entraîné, ViT pré-entraîné. Qu'apporte le transfert d'apprentissage, et à quel prix ?

In [ ]:
train_loader, validation_loader = get_dataloaders(images, labels, batch_size=64)
trainer = make_trainer(max_epochs=5, name="vit")
trainer.fit(classifier, train_loader, validation_loader)
plot_metrics(trainer)

In [ ]:
results = trainer.test(classifier, dataloaders=test_loader)
test_accuracies["ViT gelé"] = results[0]["test_accuracy"]

test_pred = predict_probabilities(trainer, classifier, test_images).argmax(dim=-1)
analyze_preds(test_pred, test_labels)

for name, accuracy in test_accuracies.items():
  print(f"{name} : {accuracy:.3f}")

#### Solution

- Les deux modèles pré-entraînés dépassent nettement le CNN entraîné depuis zéro (93 à 94 % en test, contre 84 %), en 5 époques au lieu de 15 : c'est tout l'intérêt du transfert d'apprentissage quand on a peu de données (14 000 images ici, contre plus d'un million pour ImageNet et 14 millions pour ImageNet-21k).
- Entre EfficientNetV2-B0 et le ViT, l'écart est faible, mais le ViT est bien plus gros : 86 millions de paramètres contre 5,9 millions, et chaque prédiction coûte en proportion, comme son entraînement : ses 5 époques durent autant que les 15 du CNN. Le fine-tuning du backbone apporte en général les derniers points.
- Les confusions restantes sont les mêmes pour les trois modèles : glacier et montagne, bâtiments et rue.